# Task 4 – Teil 4a: Erneuerbare Energien – Anschluss von Windpark und PV-Anlage, Ortsnetz mit PV-Haushalten

**Ausgangspunkt:** das geregelte Netz aus Teil 3b (CIGRE-MV ohne DER, CIGRE-LV-Ortsnetz an Bus 12 mit Standardtrafos, rONT-Regler je Teilnetz und Backup-Plan für das MV-Netz). Voraussetzung: `Task4-3_Ortsnetz_Auslegung.ipynb` und `Task4-3b_Regelung.ipynb` wurden ausgeführt.

**Szenarien des Arbeitsauftrags:**

1. **Windpark:** 15 WEA vom Typ Vestas V90-2.0 MW, zusammen **30 MW**, an einem 20-kV-Bus
2. **Ortsnetz mit PV-Haushalten:** Ein Teil der Haushalte im Ortsnetz erhält die BDEW-Profile P25 (Haushalt mit PV) bzw. S25 (Haushalt mit PV und Speicher)
3. **Großflächen-PV:** Freiflächenanlage mit **10 MWp** an einem 20-kV-Bus

**Vorgehen:**

1. Windpark und PV-Anlage werden **einzeln an jeden 20-kV-Bus** (Bus 1 bis Bus 14) angeschlossen und über 12 typische Wochen simuliert.
2. Ausgeschlossen wird jeder Bus, an dem eine Spannung unter 0,90 pu bzw. über 1,10 pu oder eine Auslastung über 100 % auftritt (wie in Teil 2, für Erzeugung um die Obergrenze ergänzt).
3. Alle **Kombinationen** der verbleibenden Busse für Windpark und PV-Anlage werden gemeinsam simuliert; nach denselben Kriterien wird die beste Kombination bestimmt.
4. Szenario 2 (Ortsnetz mit PV-Haushalten) wird einzeln simuliert und mit dem Normalbetrieb verglichen.

Alle Rechnungen mit allen Trennschaltern offen (Normalbetrieb) und aktiver Regelung aus Teil 3b. Die Untersuchung aller Schalterstellungen und Kombinationen der drei Szenarien folgt im nächsten Notebook.

## Importe

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import simulation as sim
import szenarien as sz
import lvnetz
import regelung as rg
import bewertung as bw
import plots
from elements import Diagram, Profile, Generator
from profiles import Month

import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=plt.cm.tab20.colors)

sim.ONLY_RUN_IF_CHANGED = True
PLAN_MARKER = [("orange", 0.95, "Planungsband 0,95 pu", "--"), ("orange", 1.05, "Planungsband 1,05 pu", "--")]

ausgelegt = sim.loadSimulation("w4_t3_ausgelegt")        # Teil 3: ausgelegtes Netz, alle Trennschalter offen
normal = sim.loadSimulation("w4_t3_geregelt")            # Teil 3b: Normalbetrieb mit Regelung (Referenz)
MV_BUSES = [f"Bus {i}" for i in range(1, 15)]

Neues Modul `bewertung.py`: Kennwerte, Ausschluss und Rangfolge von Anschlussvarianten nach einheitlichen Kriterien. `regelung.standard_regler(net)` liefert die Regler aus Teil 3b (drei rONT-Regler, Backup-Plan) für jedes Netz mit Ortsnetz.

## 1 Erzeugungsanlagen

| Anlage | Modell im Netz | Leistung | Profil | Leistungsfaktor |
|---|---|---|---|---|
| Windpark | ein Erzeuger `Windpark` (sgen) für 15 gleiche Anlagen | 15 × 2,0 MW = 30 MW | Kennlinie Vestas V90-2.0 MW aus Wind auf Nabenhöhe 105 m (`Profile.WIND`, Datengrundlage 1.2) | cos φ = 1 |
| PV-Freiflächenanlage | ein Erzeuger `PV-Freifläche` (sgen) | 10 MWp | Wetter, Modul Trina Vertex S+ (`Profile.PV`, Datengrundlage 1.3) | cos φ = 1 |

- **Windpark:** Der Arbeitsauftrag gibt 15 WEA vor; Anlagentyp und Kennlinie sind in der Datengrundlage beschrieben. Alle 15 Anlagen erhalten dieselbe Windgeschwindigkeit und werden deshalb als ein Erzeuger mit 30 MW abgebildet (gleichzeitige Einspeisung, keine Abschattung im Park; eher ungünstig für das Netz).
- **PV-Anlage:** PV-Freiflächenanlagen sind Anlagen ab 1 MWp; die 18 Zuschläge der Innovationsausschreibung im April 2021 hatten zusammen 258 MW PV-Leistung, also im Mittel rund 14 MWp [22]. 10 MWp sind damit eine typische Größe.
- **Blindleistung:** zunächst cos φ = 1 (reine Wirkleistungseinspeisung). Ein Vergleich mit Blindleistungsbereitstellung folgt bei den Szenarien.
- **Vorzeichen:** Erzeuger speisen mit `p_mw > 0` ein (Erzeugerzählpfeilsystem in pandapower).

In [3]:
def windpark(bus):
    return Generator("Windpark", bus=bus, profile=Profile.WIND, p_mw=30.0)

def pv_anlage(bus):
    return Generator("PV-Freifläche", bus=bus, profile=Profile.PV, kwp=10_000)

def mit_erzeugern(name, erzeuger):
    """Kopie des ausgelegten Netzes mit den angegebenen Erzeugern, gerechnet mit der Regelung aus Teil 3b."""
    s = ausgelegt.copy(name, overwrite=True)
    for e in erzeuger:
        s.createElement(e)
    s.runSimulation(controllers=rg.standard_regler(s.net))
    return s

def auswertung(sims, titel):
    werte, orte = bw.kennwerte(sims)
    tab, kandidaten, immer = bw.ausschluss(werte)
    print(f"{titel}: verbleibend {kandidaten or 'keine'}")
    if immer:
        print("Bei allen Varianten verletzt (für die Auswahl ausgeklammert):", immer)
    return werte, orte, tab, kandidaten

## 2 Windpark: Anschluss an jeden 20-kV-Bus

In [4]:
wind = {b: mit_erzeugern(f"w4_wind_{b.replace(' ', '')}", [windpark(b)]) for b in MV_BUSES}

Simulation 'w4_wind_Bus1': 2016 Lastflüsse in 121 s, nicht konvergiert: 0
Simulation 'w4_wind_Bus2': 2016 Lastflüsse in 132 s, nicht konvergiert: 0
Simulation 'w4_wind_Bus3': 2016 Lastflüsse in 147 s, nicht konvergiert: 0
Simulation 'w4_wind_Bus4': 2016 Lastflüsse in 130 s, nicht konvergiert: 0
Simulation 'w4_wind_Bus5': 2016 Lastflüsse in 143 s, nicht konvergiert: 0


KeyboardInterrupt: 

In [ ]:
w_werte, w_orte, w_tab, w_kand = auswertung(wind, "Windpark")
cols = ["U max gesamt [pu]", "U max MS [pu]", "U max NS [pu]", "U min gesamt [pu]", "Leitung max MS [%]",
        "Leitung max NS [%]", "Trafo max HS/MS [%]", "Ortsnetztrafo max [%]", "Stunden Rückspeisung 110 kV",
        "max. Rückspeisung 110 kV [MW]"]
display(plots.table(w_werte[cols].astype(float), decimals=3))
display(w_orte[["U max gesamt [pu]", "Leitung max MS [%]", "Trafo max HS/MS [%]"]])
w_tab

In [ ]:
w_rang, w_best = bw.rangfolge(w_werte, w_kand)
display(w_rang)
print("Bester Einzelanschluss Windpark:", w_best)

## 3 PV-Freiflächenanlage: Anschluss an jeden 20-kV-Bus

In [ ]:
pv = {b: mit_erzeugern(f"w4_pv_{b.replace(' ', '')}", [pv_anlage(b)]) for b in MV_BUSES}

In [ ]:
p_werte, p_orte, p_tab, p_kand = auswertung(pv, "PV-Anlage")
display(plots.table(p_werte[cols].astype(float), decimals=3))
display(p_orte[["U max gesamt [pu]", "Leitung max MS [%]", "Trafo max HS/MS [%]"]])
p_tab

In [ ]:
p_rang, p_best = bw.rangfolge(p_werte, p_kand)
display(p_rang)
print("Bester Einzelanschluss PV-Anlage:", p_best)

**Ergebnis der Einzeltests:**

- **Windpark (30 MW):** Nur an den beiden Trafo-Sammelschienen **Bus 1 und Bus 12** ist ein Anschluss möglich. An allen anderen Bussen werden die Kabel bzw. Freileitungen in 848 bis 1026 der 2016 Stunden über 100 % ausgelastet (ein CIGRE-Kabel trägt nur etwa 5 MVA); ab Bus 3 steigt die Spannung zusätzlich in 446 bis 689 Stunden über 1,10 pu. An Bus 2 wird nur die Leitung überlastet, weil Bus 2 noch nahe an der Sammelschiene liegt.
- **PV-Anlage (10 MWp):** Ebenfalls nur **Bus 1 und Bus 12**. An allen anderen Bussen sind die Leitungen in 24 bis 195 Stunden überlastet (Sommermittage); die Spannung bleibt unter 1,10 pu, weil die PV-Spitze mit einer mittleren Last zusammenfällt.
- **Bus 1 oder Bus 12:** An den Sammelschienen wirkt der Erzeuger nur auf den jeweiligen 110/20-kV-Trafo; Leitungen und Spannungen im Netz ändern sich kaum. Beim Windpark ist **Bus 1** besser: Trafo 0-1 erreicht höchstens 88,5 %, Trafo 0-12 bei Anschluss an Bus 12 dagegen 92,3 %, weil dort zusätzlich das Ortsnetz hängt und die Last kleiner ist. Bei der PV-Anlage sind beide Busse nach allen Kriterien gleich (die Höchstwerte treten am Winterabend auf, wenn die PV nicht einspeist); den Ausschlag geben die geringeren Verluste an **Bus 1** (158,8 statt 159,5 MWh in 12 Wochen).
- **Rückspeisung:** Der Windpark speist in 233 der 2016 Stunden mehr ein, als im Netz verbraucht wird; bis zu 16,1 MW fließen dann ins 110-kV-Netz zurück. Die PV-Anlage allein führt nie zu Rückspeisung, weil die Last tagsüber immer über 10 MW liegt.

## 4 Kombinationen von Windpark und PV-Anlage

Alle Kombinationen der in Abschnitt 2 und 3 verbleibenden Busse werden mit beiden Anlagen gleichzeitig simuliert und nach denselben Kriterien bewertet.

In [ ]:
kombi = {}
for wb in w_kand:
    for pb in p_kand:
        name = f"Wind {wb} + PV {pb}"
        kombi[name] = mit_erzeugern(f"w4_kombi_W{wb.split()[1]}_P{pb.split()[1]}", [windpark(wb), pv_anlage(pb)])
print(len(kombi), "Kombinationen")

In [ ]:
k_werte, k_orte, k_tab, k_kand = auswertung(kombi, "Kombinationen")
display(plots.table(k_werte[cols].astype(float), decimals=3))
display(k_tab)
k_rang, k_best = bw.rangfolge(k_werte, k_kand)
display(k_rang)
print("Beste Kombination:", k_best)

In [ ]:
if k_best:
    best = kombi[k_best]
    best.createDiagram(Diagram.NETWORK, time=["max", "max_loading", "min"], title=f"Beste Kombination: {k_best}")
    best.createDiagram(Diagram.POWER_FLOW, months=[Month.JANUARY, Month.JULY], title=f"Leistungsfluss – {k_best}")

**Ergebnis der Kombinationen:** Alle vier Kombinationen von Bus 1 und Bus 12 erfüllen die Kriterien. Die Kombinationen mit dem Windpark an Bus 1 sind besser (Rangsumme 10 statt 14), weil Trafo 0-1 mit 88,5 % weniger ausgelastet wird als Trafo 0-12 mit 92,3 %. Zwischen PV an Bus 1 und PV an Bus 12 gibt es bei den Höchstwerten keinen Unterschied; die geringeren Verluste entscheiden für **Windpark an Bus 1 und PV-Anlage an Bus 12** (151,6 MWh gegenüber 153,0 MWh). Die Verteilung auf beide Sammelschienen ist auch physikalisch sinnvoll: Jeder Trafo nimmt einen Teil der Einspeisung auf, und die Verluste sinken gegenüber dem Normalbetrieb (162,8 MWh), weil ein Teil der Last direkt vor Ort gedeckt wird. Zusammen speisen Windpark und PV-Anlage in 263 Stunden ins 110-kV-Netz zurück, höchstens 16,1 MW.

Die Unterschiede zwischen den Kombinationen sind klein. Das liegt daran, dass beide Anlagen an den Trafo-Sammelschienen hängen, deren Spannung über die starren Transformatoren direkt vom 110-kV-Netz gehalten wird.

## 5 Ortsnetz mit PV-Haushalten (Szenario 2)

Im Ortsnetz erhält ein Teil der Haushalte die BDEW-Profile für Haushalte mit PV (P25) bzw. mit PV und Speicher (S25). Die Profile beschreiben den **Netzbezug** dieser Haushalte; die PV-Einspeisung ist nicht enthalten und wird bei den BDEW-Profilen über eigene Erzeugungsprofile abgebildet [1, S. 4–5]. Das Szenario zeigt damit, wie sich der Strombezug der Haushalte durch PV und Speicher verändert; eine Rückspeisung aus den Haushalten bildet es nicht ab.

**Aufteilung:** Jede Haushaltslast im Ortsnetz (`LV Load R…`) wird in drei Teillasten aufgeteilt (H25, P25, S25). Jede Teillast erhält den entsprechenden Anteil der ursprünglichen Höchstleistung, weil sich die Winter-Abendspitze eines Haushalts durch PV kaum ändert. Der Jahresverbrauch folgt daraus wie bisher über die Höchstleistung des jeweiligen Profils.

| Variante | Haushalte mit PV (P25) | Haushalte mit PV und Speicher (S25) | Haushalte ohne PV (H25) | Quelle |
|---|---|---|---|---|
| heute | 8,3 % | 6,7 % | 85 % | KfW: fast 15 % der Haushalte mit PV, 6,7 % mit Batteriespeicher [23] |
| Zukunft | 10 % | 40 % | 50 % | Annahme: 50 % der Haushalte mit PV; fast 80 % der neuen privaten Aufdachanlagen mit Speicher [24] |

Ein Speicher wird praktisch immer zusammen mit einer PV-Anlage betrieben; die 6,7 % mit Speicher sind deshalb eine Teilmenge der 15 % mit PV.

In [ ]:
VARIANTEN = {"heute": (0.083, 0.067), "Zukunft": (0.10, 0.40)}
ortsnetz = {}
for name, (a_pv, a_sp) in VARIANTEN.items():
    s = ausgelegt.copy(f"w4_ortsnetz_pv_{name}", overwrite=True)
    aufteilung = lvnetz.prosumer(s, a_pv, a_sp)
    s.runSimulation(controllers=rg.standard_regler(s.net))
    ortsnetz[name] = s
    print(name); display(aufteilung.round(4))

In [ ]:
vergleich = {"Normalbetrieb": normal, **{f"PV-Haushalte {k}": s for k, s in ortsnetz.items()}}
o_werte, o_orte = bw.kennwerte(vergleich)
lv_last = {k: s.results.load_p_mw[[c for c in s.results.load_p_mw if c.startswith("LV Load R")]].sum().sum() for k, s in vergleich.items()}
o_werte["Energie Haushalte Ortsnetz [MWh]"] = pd.Series(lv_last)
o_werte["Stufenschritte R"] = [rg.schaltvorgaenge(s).get("Stufenschritte LV Trafo R0-R1", 0) for s in vergleich.values()]
plots.table(o_werte[["U min NS [pu]", "U max NS [pu]", "Stunden U < 0,95 NS", "Ortsnetztrafo max [%]",
                     "Energie Haushalte Ortsnetz [MWh]", "Stufenschritte R"]].astype(float).T, decimals=3)

In [ ]:
sim.compare(list(vergleich.values()), Diagram.APPARENT_POWER, ["LV Trafo R0-R1"], months=[Month.JANUARY, Month.JULY],
            title="Scheinleistung Ortsnetztrafo Wohngebiet")
sim.compare(list(vergleich.values()), Diagram.VOLTAGE, ["LV Bus R15"], months=[Month.JULY], markers=PLAN_MARKER,
            title="Spannung am Strangende des Wohngebiets (LV Bus R15)")

**Ergebnis Ortsnetz mit PV-Haushalten:**

| Kennwert | Normalbetrieb | PV-Haushalte heute | PV-Haushalte Zukunft |
|---|---|---|---|
| Energie Haushalte im Ortsnetz (12 Wochen) | 390 MWh | 366 MWh (−6 %) | 292 MWh (−25 %) |
| Ortsnetztrafo Wohngebiet, höchste Auslastung im Juli | 45,1 % | 38,8 % | 23,7 % |
| Ortsnetztrafo Wohngebiet, höchste Auslastung im Januar | 66,8 % | 65,8 % | 63,3 % |
| Stufenschritte rONT Wohngebiet | 92 | 80 | 58 |
| niedrigste / höchste NS-Spannung | 0,950 / 1,027 pu | 0,950 / 1,026 pu | 0,951 / 1,026 pu |

- PV und Speicher senken den **Strombezug** der Haushalte, vor allem im Sommer: Im Zukunftsszenario ist der Ortsnetztrafo im Juli nur noch halb so stark ausgelastet. Die **Winterspitze** bleibt fast gleich, weil PV am Winterabend nicht einspeist und die Speicher im Winter kaum geladen werden. Für die Auslegung der Betriebsmittel ändert sich deshalb wenig.
- Der rONT muss seltener eingreifen (58 statt 92 Stufenschritte im Wohngebiet), weil die Spannung am Strangende im Sommer weniger absinkt. Im Gewerbenetz ändert sich nichts.
- Weil die BDEW-Profile P25 und S25 nur den Netzbezug enthalten, entsteht in diesem Szenario keine Rückspeisung aus den Haushalten und keine Spannungsanhebung im Ortsnetz. Das ist eine Grenze des gewählten Modells.

## 6 Ergebnis

- **Anschluss:** Windpark (30 MW) und PV-Freiflächenanlage (10 MWp) lassen sich im CIGRE-MV-Netz nur an den Sammelschienen der 110/20-kV-Transformatoren anschließen (Bus 1 oder Bus 12). In den Abgängen überlasten sie die Leitungen, die nur für etwa 5 MVA ausgelegt sind. In der Praxis werden Anlagen dieser Größe ebenfalls über ein eigenes Kabel direkt an die Sammelschiene des Umspannwerks angeschlossen.
- **Beste Kombination:** Windpark an **Bus 1**, PV-Anlage an **Bus 12**. Damit wird jeder Transformator nur mit einer Anlage zusätzlich belastet, Trafo 0-1 erreicht höchstens 88,5 %, und die Verluste sind am geringsten.
- **Ortsnetz:** Die Profile für Haushalte mit PV und Speicher senken den Strombezug im Sommer deutlich, ändern aber die Winterspitze kaum.
- **Nächster Schritt:** Mit diesen Anschlussbussen werden die drei Szenarien einzeln und kombiniert für alle Schalterstellungen S1–S3 untersucht, einschließlich hoher und niedriger Einspeise- und Lastsituationen, Blindleistung und Lastflussumkehr.

### Quellen

[1] BDEW, „Hinweise zu den aktualisierten Standardlastprofilen Strom“, Anwendungshilfe, Berlin, 17.03.2025. Online: https://bdew.de/media/documents/2025-03-17_AWH_Aktualisierte_SLP_Strom_2025_Ver%C3%B6ffentlichung.pdf

[22] C. Kost et al., „Stromgestehungskosten Erneuerbare Energien“, Fraunhofer ISE, Freiburg, Juni 2021. Online: https://www.ise.fraunhofer.de/content/dam/ise/de/documents/publications/studies/DE2021_ISE_Studie_Stromgestehungskosten_Erneuerbare_Energien.pdf

[23] KfW Research, „Fast 13 Millionen Haushalte nutzen Energiewendetechnologien“, Pressemitteilung, 12.09.2024 (KfW-Energiewendebarometer). Online: https://www.kfw.de/%C3%9Cber-die-KfW/Newsroom/Aktuelles/Pressemitteilungen-Details_821696.html

[24] Fraunhofer ISE, „Photovoltaics Report“, Freiburg. Online: https://www.ise.fraunhofer.de/content/dam/ise/de/documents/publications/studies/Photovoltaics-Report.pdf

*Bei [22] und [24] Seiten bzw. Folien ergänzen und die Ausgabe von [24] prüfen.*